<a href="https://colab.research.google.com/github/moy850/avionica-reto-2026/blob/main/flight_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# RETO DE AVIONICA
# GENERADOR DE VUELO COMPLETO

import csv
import random
import math

# CONFIGURACION
dt = 0.1
gravedad = 9.81
altitud = 0.0
velocidad = 0.0
tiempo = 0.0

# Listas para guardar la telemetria
datos = []

def registrar_dato(t, h, a):
    # Ruido gaussiano del barometro
    ruido = random.gauss(0, 0.5)
    altitud_medida = max(0, h + ruido)

    # Gradiente termico: 6.5 grados por km
    temperatura = 20 - (6.5 * h / 1000)

    datos.append([
        round(t * 1000),
        round(altitud_medida, 3),
        round(a, 3),
        round(temperatura, 3)
    ])

# FASE 1: EN RAMPA (2 segundos)
for i in range(20):
    tiempo = i * dt
    registrar_dato(tiempo, 0, 9.81)

# FASE 2: PROPULSION (1.5 segundos)
altitud = 0
velocidad = 0

for i in range(1, 16):
    altitud = altitud + velocidad * dt + 0.5 * 60 * dt**2
    velocidad = velocidad + 60 * dt
    tiempo = 2 + i * dt

    registrar_dato(tiempo, altitud, 69.81)

# FASE 3: VUELO BALISTICO
while velocidad > 0:
    altitud = altitud + velocidad * dt - 0.5 * gravedad * dt**2
    velocidad = velocidad - gravedad * dt
    tiempo = tiempo + dt

    registrar_dato(tiempo, altitud, 0)

# FASE 4: PARACAIDAS
while altitud > 0:
    altitud = altitud - 6 * dt
    tiempo = tiempo + dt

    if altitud < 0:
        altitud = 0

    registrar_dato(tiempo, altitud, 9.81)

# FASE 5: EN TIERRA (3 segundos)
for i in range(1, 31):
    tiempo = tiempo + dt
    registrar_dato(tiempo, 0, 9.81)

# CREAR ARCHIVO CSV
with open("flight_log.csv", "w", newline="") as archivo:
    escritor = csv.writer(archivo)

    escritor.writerow([
        "time_ms",
        "altitude_m",
        "accel_z_ms2",
        "temp_c"
    ])

    # Insertar 3 lineas corruptas
    for i, fila in enumerate(datos):
        if i == 30:
            escritor.writerow(["", "ERROR", "69.81", "19.9"])
        if i == 60:
            escritor.writerow(["ERROR", "450", "0", "17"])
        if i == 90:
            escritor.writerow(["5000", "300", "0"])

        escritor.writerow(fila)

print("SIMULACION COMPLETADA")
print("Muestras validas:", len(datos))
print("Archivo creado: flight_log.csv")
print("Altitud maxima:", round(max(f[1] for f in datos), 2), "m")
print("Duracion:", round(datos[-1][0] / 1000, 2), "segundos")

SIMULACION COMPLETADA
Muestras validas: 958
Archivo creado: flight_log.csv
Altitud maxima: 480.82 m
Duracion: 95.8 segundos


In [ ]:
from google.colab import files

files.download("flight_log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Mostrar las primeras 10 muestras

with open("flight_log.csv", "r") as archivo:
    for i in range(10):
        print(archivo.readline().strip())

time_ms,altitude_m,accel_z_ms2,temp_c
0,0,9.81,20.0
100,0,9.81,20.0
200,0,9.81,20.0
300,0.139,9.81,20.0
400,0,9.81,20.0
500,0.527,9.81,20.0
600,0,9.81,20.0
700,0,9.81,20.0
800,0,9.81,20.0


In [ ]:
# PASO 11: LIMPIEZA DE TELEMETRIA

import csv
import math

datos_limpios = []
lineas_descartadas = 0

with open("flight_log.csv", "r") as archivo:

    lector = csv.reader(archivo)

    encabezado = next(lector)

    for fila in lector:

        try:
            # Comprobar que tenga 4 columnas
            if len(fila) != 4:
                raise ValueError("Columnas incorrectas")

            # Convertir los datos a numeros
            tiempo = float(fila[0])
            altitud = float(fila[1])
            aceleracion = float(fila[2])
            temperatura = float(fila[3])

            # Comprobar que sean valores validos
            valores = [tiempo, altitud, aceleracion, temperatura]

            if not all(math.isfinite(x) for x in valores):
                raise ValueError("Dato no valido")

            # Guardar el registro limpio
            datos_limpios.append([
                tiempo / 1000,
                altitud,
                aceleracion,
                temperatura
            ])

        except (ValueError, TypeError):
            lineas_descartadas += 1

print("PROCESAMIENTO INICIAL")
print("Registros validos:", len(datos_limpios))
print("Lineas descartadas:", lineas_descartadas)

print("\nPrimeros 5 registros:")
for fila in datos_limpios[:5]:
    print(fila)

PROCESAMIENTO INICIAL
Registros validos: 958
Lineas descartadas: 3

Primeros 5 registros:
[0.0, 0.319, 9.81, 20.0]
[0.1, 0.0, 9.81, 20.0]
[0.2, 0.0, 9.81, 20.0]
[0.3, 0.0, 9.81, 20.0]
[0.4, 0.0, 9.81, 20.0]


In [ ]:
#paso 12 filtro de promedio movil
N = 5 # cantidad de muestras para calcular el promedio
altitudes_filtradas = []

for i in range(len(datos_limpios)):
    numero = max(0, i - N + 1)
    ventana = datos_limpios[numero: i + 1]
    promedio = sum(fila[1] for fila in ventana) / len(ventana)
    altitudes_filtradas.append(promedio)

print("FILTRO APLICADO")
print("tamaño de pantalla:", N )
print("\nComparacion de las primeras 10 mediciones:")
for i in range(10):
    print(" original datos:",
        round(datos_limpios[i][1], 2),
        "| filtrada:",
        round(altitudes_filtradas[i], 2)
    )

FILTRO APLICADO
tamaño de pantalla: 5

Comparacion de las primeras 10 mediciones:
 original datos: 0.32 | filtrada: 0.32
 original datos: 0.0 | filtrada: 0.16
 original datos: 0.0 | filtrada: 0.11
 original datos: 0.0 | filtrada: 0.08
 original datos: 0.0 | filtrada: 0.06
 original datos: 0.34 | filtrada: 0.07
 original datos: 0.0 | filtrada: 0.07
 original datos: 0.0 | filtrada: 0.07
 original datos: 0.0 | filtrada: 0.07
 original datos: 0.2 | filtrada: 0.11
